# AuraGateway vLLM CUDA 12.9 wheelhouse materializer v1


In [ ]:
from __future__ import annotations

import base64
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request
import zlib
from pathlib import Path, PurePosixPath

NOTEBOOK_NAME = "auragateway-cu129-wheelhouse-materializer-v1"
OUTPUT_DIRECTORY_NAME = "auragateway_vllm_cu129_wheelhouse_v1"
OUTPUT_ROOT = Path("/kaggle/working") / OUTPUT_DIRECTORY_NAME
WHEELHOUSE = OUTPUT_ROOT / "wheels"

VLLM_RELEASE = "0.19.1"
VLLM_DISTRIBUTION = "0.19.1"
VLLM_RELEASE_API = (
    "https://api.github.com/repos/vllm-project/vllm/releases/tags/v0.19.1"
)
VLLM_ASSET_NAME = "vllm-0.19.1-cp38-abi3-manylinux_2_31_x86_64.whl"
VLLM_ASSET_SHA256 = (
    "71a87f46cafab4489c69a5c5c83b870d0235e5694d8222303d460576293dc719"
)
PYPI_INDEX = "https://pypi.org/simple"
PYTORCH_INDEX = "https://download.pytorch.org/whl/cu129"
ALLOWED_DOWNLOAD_HOSTS = {
    "download.pytorch.org",
    "download-r2.pytorch.org",
    "files.pythonhosted.org",
    "github.com",
    "pypi.nvidia.com",
}
FIXED_REQUIREMENTS = (
    "torch==2.10.0+cu129",
    "torchaudio==2.10.0+cu129",
    "torchvision==0.25.0+cu129",
    "transformers==5.5.3",
)
RESOLUTION_LOCK_SHA256 = "1575538b0a412c9b030fc95ccada0f0527553b76f06ef6b2b72904e61c84870c"
RESOLUTION_LOCK_B64 = (
    "..."
)
EXPECTED_PACKAGE_COUNT = 176
CREDENTIAL_ENV_NAMES = (
    "ANTHROPIC_API_KEY",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "GOOGLE_API_KEY",
    "HF_TOKEN",
    "HUGGING_FACE_HUB_TOKEN",
    "OPENAI_API_KEY",
    "OPENROUTER_API_KEY",
)


def canonical_json(payload: object) -> str:
    return json.dumps(payload, ensure_ascii=True, separators=(",", ":"), sort_keys=True)


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def load_resolution_lock() -> dict[str, object]:
    payload = json.loads(zlib.decompress(base64.b64decode(RESOLUTION_LOCK_B64)).decode("utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError("embedded resolution lock is not one JSON object")
    canonical = canonical_json(payload)
    if sha256_text(canonical + "\n") != RESOLUTION_LOCK_SHA256:
        raise RuntimeError("embedded resolution lock SHA-256 drifted")
    if payload.get("package_count") != EXPECTED_PACKAGE_COUNT:
        raise RuntimeError("embedded resolution lock package count drifted")
    return payload


def resolved_artifact_identity(
    *,
    name: str,
    version: str,
    url: str,
    digest: str,
) -> dict[str, str]:
    parsed = urllib.parse.urlsplit(url)
    return {
        "normalized_name": name.lower().replace("_", "-"),
        "version": version,
        "hostname": parsed.hostname or "missing",
        "artifact_filename": PurePosixPath(parsed.path).name or "missing",
        "url_sha256": sha256_text(url),
        "sha256": digest,
    }


def sanitized_excerpt(text: str) -> str:
    return text[-12000:].replace("/kaggle/working", "<working>")


def run(argv: list[str], *, timeout: float) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(
        argv,
        check=False,
        capture_output=True,
        text=True,
        timeout=timeout,
        env={**os.environ, "PIP_DISABLE_PIP_VERSION_CHECK": "1"},
    )
    if result.returncode != 0:
        raise RuntimeError(
            canonical_json(
                {
                    "command_role": argv[2] if len(argv) > 2 else argv[0],
                    "returncode": result.returncode,
                    "stdout_excerpt": sanitized_excerpt(result.stdout),
                    "stderr_excerpt": sanitized_excerpt(result.stderr),
                }
            )
        )
    return result


def validate_download_url(
    value: object,
    *,
    distribution_name: str,
) -> str:
    normalized_distribution = distribution_name.lower().replace("_", "-")[:120]
    if not isinstance(value, str):
        raise RuntimeError(
            canonical_json(
                {
                    "distribution_name": normalized_distribution,
                    "failure_code": "RESOLVED_ARTIFACT_URL_MISSING",
                }
            )
        )
    parsed = urllib.parse.urlsplit(value)
    if (
        parsed.scheme != "https"
        or parsed.hostname not in ALLOWED_DOWNLOAD_HOSTS
        or parsed.username is not None
        or parsed.password is not None
        or parsed.fragment
    ):
        raise RuntimeError(
            canonical_json(
                {
                    "distribution_name": normalized_distribution,
                    "failure_code": "RESOLVED_ARTIFACT_URL_NOT_ALLOWED",
                    "hostname": parsed.hostname or "missing",
                }
            )
        )
    return value


def release_asset() -> tuple[str, str, str | None]:
    request = urllib.request.Request(
        VLLM_RELEASE_API,
        headers={
            "Accept": "application/vnd.github+json",
            "User-Agent": NOTEBOOK_NAME,
            "X-GitHub-Api-Version": "2022-11-28",
        },
    )
    with urllib.request.urlopen(request, timeout=60.0) as response:
        payload = json.loads(response.read().decode("utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError("vLLM release metadata is not one JSON object")
    if payload.get("tag_name") != f"v{VLLM_RELEASE}":
        raise RuntimeError("vLLM release tag drifted")
    if payload.get("draft") is not False or payload.get("prerelease") is not False:
        raise RuntimeError("vLLM release must be published and non-prerelease")
    assets = payload.get("assets")
    if not isinstance(assets, list):
        raise RuntimeError("vLLM release assets are missing")
    candidates = [
        item
        for item in assets
        if isinstance(item, dict) and item.get("name") == VLLM_ASSET_NAME
    ]
    if len(candidates) != 1:
        raise RuntimeError("expected the exact official vLLM CUDA 12.9 x86_64 wheel")
    candidate = candidates[0]
    name = str(candidate["name"])
    url = validate_download_url(
        candidate.get("browser_download_url"),
        distribution_name="vllm",
    )
    raw_digest = candidate.get("digest")
    if not isinstance(raw_digest, str) or not raw_digest.startswith("sha256:"):
        raise RuntimeError("vLLM release asset lacks a SHA-256 identity")
    digest = raw_digest.removeprefix("sha256:")
    if digest != VLLM_ASSET_SHA256:
        raise RuntimeError("vLLM release asset SHA-256 drifted")
    return name, url, digest


def archive_sha256(archive_info: dict[str, object]) -> str:
    hash_value = archive_info.get("hash")
    if isinstance(hash_value, str) and hash_value.startswith("sha256="):
        digest = hash_value.removeprefix("sha256=")
        if len(digest) == 64:
            return digest
    hashes = archive_info.get("hashes")
    if isinstance(hashes, dict):
        sha256_value = hashes.get("sha256")
        if isinstance(sha256_value, str) and len(sha256_value) == 64:
            return sha256_value
    raise RuntimeError("resolved artifact lacks one SHA-256 identity")


present_credentials = [name for name in CREDENTIAL_ENV_NAMES if os.environ.get(name)]
if present_credentials:
    raise RuntimeError("credential-bearing environment variables are prohibited")

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True)
WHEELHOUSE.mkdir()

asset_name, asset_url, release_digest = release_asset()
vllm_requirement = f"vllm @ {asset_url}"
top_level_requirements = (vllm_requirement, *FIXED_REQUIREMENTS)

requirements_in = OUTPUT_ROOT / "requirements.in"
requirements_in.write_text(
    "\n".join(top_level_requirements) + "\n",
    encoding="utf-8",
)

with tempfile.TemporaryDirectory(prefix="ag-vllm-resolve-") as raw_temp:
    temp_root = Path(raw_temp)
    report_path = temp_root / "resolution-report.json"
    resolution_result = run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--dry-run",
            "--ignore-installed",
            "--only-binary=:all:",
            "--report",
            str(report_path),
            "--index-url",
            PYPI_INDEX,
            "--extra-index-url",
            PYTORCH_INDEX,
            "-r",
            str(requirements_in),
        ],
        timeout=1800.0,
    )
    resolution_transfer_event_count = resolution_result.stdout.count("Downloading ")
    report = json.loads(report_path.read_text(encoding="utf-8"))

install_records = report.get("install")
if not isinstance(install_records, list) or not install_records:
    raise RuntimeError("pip resolution report contains no install records")

resolution_lock = load_resolution_lock()
expected_records_raw = resolution_lock.get("records")
if not isinstance(expected_records_raw, list):
    raise RuntimeError("embedded resolution lock records are missing")
expected_records = {
    str(record["normalized_name"]): record
    for record in expected_records_raw
    if isinstance(record, dict) and isinstance(record.get("normalized_name"), str)
}
if len(expected_records) != EXPECTED_PACKAGE_COUNT:
    raise RuntimeError("embedded resolution lock contains duplicate or invalid records")

locked_records: list[dict[str, str]] = []
actual_identities: dict[str, dict[str, str]] = {}
for item in install_records:
    if not isinstance(item, dict):
        raise RuntimeError("pip resolution report entry is invalid")
    metadata = item.get("metadata")
    download_info = item.get("download_info")
    if not isinstance(metadata, dict) or not isinstance(download_info, dict):
        raise RuntimeError("pip resolution report entry is incomplete")
    archive_info = download_info.get("archive_info")
    if not isinstance(archive_info, dict):
        raise RuntimeError("pip resolution report archive identity is missing")
    name = str(metadata["name"])
    version = str(metadata["version"])
    url = validate_download_url(
        download_info.get("url"),
        distribution_name=name,
    )
    if name.lower().replace("_", "-") == "vllm":
        url = asset_url
        if version != VLLM_DISTRIBUTION:
            raise RuntimeError(
                f"expected vLLM distribution {VLLM_DISTRIBUTION}, observed {version}"
            )
    digest = archive_sha256(archive_info)
    if (
        name.lower().replace("_", "-") == "vllm"
        and release_digest is not None
        and digest != release_digest
    ):
        raise RuntimeError("GitHub and pip vLLM asset digests disagree")
    identity = resolved_artifact_identity(
        name=name,
        version=version,
        url=url,
        digest=digest,
    )
    normalized_name = identity["normalized_name"]
    if normalized_name in actual_identities:
        raise RuntimeError("resolved dependency closure contains duplicate distributions")
    actual_identities[normalized_name] = identity
    locked_records.append(
        {
            "name": name,
            "normalized_name": normalized_name,
            "version": version,
            "url": url,
            "sha256": digest,
        }
    )

mismatches: list[dict[str, object]] = []
for normalized_name in sorted(set(expected_records) | set(actual_identities)):
    expected = expected_records.get(normalized_name)
    actual = actual_identities.get(normalized_name)
    if expected is None:
        mismatches.append(
            {
                "distribution_name": normalized_name,
                "failure_code": "UNEXPECTED_RESOLVED_DISTRIBUTION",
            }
        )
        continue
    if actual is None:
        mismatches.append(
            {
                "distribution_name": normalized_name,
                "failure_code": "LOCKED_DISTRIBUTION_MISSING",
            }
        )
        continue
    drift = {
        field: {"expected": expected.get(field), "observed": actual.get(field)}
        for field in ("version", "hostname", "artifact_filename", "url_sha256", "sha256")
        if expected.get(field) != actual.get(field)
    }
    if drift:
        mismatches.append(
            {
                "distribution_name": normalized_name,
                "failure_code": "RESOLUTION_LOCK_DRIFT",
                "drift": drift,
            }
        )

if mismatches:
    raise RuntimeError(
        canonical_json(
            {
                "failure_code": "RESOLUTION_LOCK_MISMATCH",
                "mismatch_count": len(mismatches),
                "mismatches": mismatches[:50],
            }
        )
    )

expected_digests = [record["sha256"] for record in locked_records]
if len(expected_digests) != len(set(expected_digests)):
    raise RuntimeError("resolved dependency closure contains duplicate artifact hashes")

locked_records.sort(key=lambda record: record["normalized_name"])

resolution_lock_path = OUTPUT_ROOT / "resolution_lock.json"
resolution_lock_path.write_text(
    canonical_json(resolution_lock) + "\n",
    encoding="utf-8",
)
if sha256_file(resolution_lock_path) != RESOLUTION_LOCK_SHA256:
    raise RuntimeError("written resolution lock identity drifted")

materialization_lock_path = OUTPUT_ROOT / "materialization.lock.txt"
materialization_lock_path.write_text(
    "".join(
        f"{record['name']} @ {record['url']} "
        f"--hash=sha256:{record['sha256']}\n"
        for record in locked_records
    ),
    encoding="utf-8",
)

runtime_lock_path = OUTPUT_ROOT / "requirements.lock.txt"
runtime_lock_path.write_text(
    "".join(
        f"{record['name']}=={record['version']} "
        f"--hash=sha256:{record['sha256']}\n"
        for record in locked_records
    ),
    encoding="utf-8",
)

run(
    [
        sys.executable,
        "-m",
        "pip",
        "download",
        "--dest",
        str(WHEELHOUSE),
        "--only-binary=:all:",
        "--require-hashes",
        "--no-deps",
        "-r",
        str(materialization_lock_path),
    ],
    timeout=5400.0,
)

wheel_paths = tuple(sorted(WHEELHOUSE.iterdir(), key=lambda path: path.name.lower()))
if len(wheel_paths) != len(locked_records):
    raise RuntimeError("downloaded wheel count does not match the resolved lock")
if any(not path.is_file() or path.suffix != ".whl" for path in wheel_paths):
    raise RuntimeError("wheelhouse must contain regular wheel files only")

observed_digests = {sha256_file(path) for path in wheel_paths}
if observed_digests != set(expected_digests):
    raise RuntimeError("downloaded wheel hashes do not match the resolved closure")

required_prefixes = (
    "vllm-0.19.1-",
    "torch-2.10.0+cu129-",
    "torchaudio-2.10.0+cu129-",
    "torchvision-0.25.0+cu129-",
    "transformers-5.5.3-",
)
for prefix in required_prefixes:
    if sum(path.name.startswith(prefix) for path in wheel_paths) != 1:
        raise RuntimeError(f"wheelhouse lacks one exact required artifact: {prefix}")

if sum(path.name == asset_name for path in wheel_paths) != 1:
    raise RuntimeError("materialized vLLM asset name drifted")

installer_source = '''
from __future__ import annotations

import argparse
import subprocess
import sys
import venv
from pathlib import Path


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--wheelhouse", type=Path, required=True)
    parser.add_argument("--lock", type=Path, required=True)
    parser.add_argument("--venv", type=Path, required=True)
    args = parser.parse_args()
    if args.venv.exists():
        raise SystemExit("target virtual environment already exists")
    venv.EnvBuilder(with_pip=True, clear=False).create(args.venv)
    python = args.venv / "bin" / "python"
    command = [
        str(python),
        "-m",
        "pip",
        "install",
        "--no-index",
        "--find-links",
        str(args.wheelhouse),
        "--require-hashes",
        "-r",
        str(args.lock),
    ]
    result = subprocess.run(command, check=False)
    return result.returncode


if __name__ == "__main__":
    sys.exit(main())
'''.lstrip()
installer_path = OUTPUT_ROOT / "install_runtime.py"
installer_path.write_text(installer_source, encoding="utf-8")

runtime_manifest = {
    "schema_version": "1.2.0",
    "runtime_id": "auragateway-vllm-cu129-runtime-v1",
    "python": "3.12",
    "cuda_variant": "cu129",
    "vllm_binary_cuda": "12.9",
    "vllm_release": VLLM_RELEASE,
    "vllm": VLLM_DISTRIBUTION,
    "vllm_asset_name": asset_name,
    "vllm_asset_sha256": next(
        record["sha256"]
        for record in locked_records
        if record["normalized_name"] == "vllm"
    ),
    "torch": "2.10.0+cu129",
    "torchaudio": "2.10.0+cu129",
    "torchvision": "0.25.0+cu129",
    "transformers": "5.5.3",
    "package_count": len(wheel_paths),
    "resolution_lock_sha256": RESOLUTION_LOCK_SHA256,
    "installation_mode": "isolated_virtual_environment",
    "network_required_for_installation": False,
    "model_requests_performed": 0,
    "qualification_claimed": False,
}
manifest_path = OUTPUT_ROOT / "runtime_manifest.json"
manifest_path.write_text(canonical_json(runtime_manifest) + "\n", encoding="utf-8")

payload_paths = (
    requirements_in,
    resolution_lock_path,
    materialization_lock_path,
    runtime_lock_path,
    installer_path,
    manifest_path,
    *wheel_paths,
)
sha_entries = [
    {
        "path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size,
    }
    for path in payload_paths
]
sha_manifest_path = OUTPUT_ROOT / "sha256_manifest.json"
sha_manifest_path.write_text(
    canonical_json({"schema_version": "1.0.0", "entries": sha_entries}) + "\n",
    encoding="utf-8",
)

receipt = {
    "schema_version": "1.2.0",
    "notebook_name": NOTEBOOK_NAME,
    "output_directory": OUTPUT_DIRECTORY_NAME,
    "package_count": len(wheel_paths),
    "total_wheel_bytes": sum(path.stat().st_size for path in wheel_paths),
    "resolution_lock_sha256": RESOLUTION_LOCK_SHA256,
    "pip_resolution_artifact_transfer_observed": resolution_transfer_event_count > 0,
    "pip_resolution_transfer_event_count": resolution_transfer_event_count,
    "pip_download_subcommand_performed": True,
    "package_installation_performed": False,
    "requirements_lock_sha256": sha256_file(runtime_lock_path),
    "materialization_lock_sha256": sha256_file(materialization_lock_path),
    "runtime_manifest_sha256": sha256_file(manifest_path),
    "sha256_manifest_sha256": sha256_file(sha_manifest_path),
    "credentials_used": False,
    "customer_data_used": False,
    "model_requests_performed": 0,
    "qualification_claimed": False,
    "materialization_status": "PASSED",
}
receipt_path = OUTPUT_ROOT / "materialization_receipt.json"
receipt_path.write_text(canonical_json(receipt) + "\n", encoding="utf-8")

print(f"output_directory={OUTPUT_ROOT}")
print(f"vllm_asset_name={asset_name}")
print(f"vllm_distribution={VLLM_DISTRIBUTION}")
print(f"package_count={len(wheel_paths)}")
print(f"resolution_lock_sha256={RESOLUTION_LOCK_SHA256}")
print(f"total_wheel_bytes={receipt['total_wheel_bytes']}")
print("materialization_status=PASSED")
print("package_installation_performed=false")
print("model_requests_performed=0")
print("qualification_claimed=false")
print("save_this_notebook_output=true")
